# Strategic News Summary Pipeline

## Load useful libraries

In [ ]:
import os
import shutil
import papermill as pm
import time

## User settings

In [ ]:
directory_output_root = os.environ['STRATEGIC_REPORTS_HOME'] + '/output/daily'
filename_json_feeds_list = os.environ['STRATEGIC_REPORTS_HOME'] + '/data/daily/feeds_ai.json'
directory_notebooks = os.environ['STRATEGIC_REPORTS_HOME'] + '/strategic_reports/daily'
number_of_seconds_to_sleep = 5

## Define output directory

In [ ]:
directory_name = filename_json_feeds_list.split('/')[-1].replace('.json', '')

In [ ]:
directory_output_path = directory_output_root + '/' + directory_name

## Create or replace output directory

In [ ]:
shutil.rmtree(directory_output_path, ignore_errors = True)
os.system('mkdir ' + directory_output_path)

## Run the RSS content retrieval notebook

In [ ]:
parameters_retrieve_rss = {
    'filename_json_feeds_list' : filename_json_feeds_list,
    'filename_output_pickled' : directory_output_path + '/feed_content.pickled',
    'filename_output_exceptions' : directory_output_path + '/exceptions_feed_pull.json'
}

In [ ]:
result = pm.execute_notebook(
   directory_notebooks + '/retrieve-rss-content.ipynb',
   directory_output_path + '/OUTPUT-retrieve-rss-content.ipynb',
   parameters = parameters_retrieve_rss,
)

In [ ]:
time.sleep(number_of_seconds_to_sleep)

## Run the summarization and tagging notebook

In [ ]:
parameters_summaries_and_tags = {
    'filename_pickled_feeds' : directory_output_path + '/feed_content.pickled',
    'filename_json_summaries_and_tags' : directory_output_path + '/summaries_and_llm_tags.json'    
}

In [ ]:
result = pm.execute_notebook(
   directory_notebooks + '/generate-summaries-and-tags.ipynb',
   directory_output_path + '/OUTPUT-generate-summaries-and-tags.ipynb',
   parameters = parameters_summaries_and_tags,
)

In [ ]:
time.sleep(number_of_seconds_to_sleep)

## Produce the summaries portion of the final report

In [ ]:
parameters_summaries_report = {
    'filename_json_summaries_and_tags' : directory_output_path + '/summaries_and_llm_tags.json',
    'filename_markdown_report_summaries_and_tags' : directory_output_path + '/report_summaries_and_llm_tags.md',
    'filename_pickled_tag_relationships' : directory_output_path + '/llm_tag_relationships.pickled',
}

In [ ]:
result = pm.execute_notebook(
    directory_notebooks + '/generate-article-summaries-report.ipynb',
    directory_output_path + '/OUTPUT-generate-article-summaries-report.ipynb',
    parameters = parameters_summaries_report,
)

In [ ]:
time.sleep(number_of_seconds_to_sleep)

## Derive the strategic conclusions

In [ ]:
parameters_strategy = {
    'filename_markdown_report_summaries_and_tags' : directory_output_path + '/report_summaries_and_llm_tags.md',
    'filename_markdown_report_strategy' : directory_output_path + '/report_strategy.md',
}

In [ ]:
result = pm.execute_notebook(
    directory_notebooks + '/strategy-derivation.ipynb',
    directory_output_path + '/OUTPUT-strategy-derivation.ipynb',
    parameters = parameters_strategy,
)

In [ ]:
time.sleep(number_of_seconds_to_sleep)